# NDVI Calculation

In [6]:
import sys
sys.path.append('/work')


In [7]:
import datetime as dt
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import rasterio
import lithops
import time
import shutil
import os
import gc
import datetime
import math
import collections
from rasterio.io import MemoryFile
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from IPython import display

import cloudbutton_geospatial.s2froms3 as s2froms3
from cloudbutton_geospatial.utils import notebook as notebook_utils
from cloudbutton_geospatial.io_utils.ndvi import get_ndvi_params, ndvi_calculation, ndvi_tile_sentinel, get_subset_raster, lonlat_to_utm, get_poly_within
from cloudbutton_geospatial.io_utils.plot import tiff_overview, plot_map

%matplotlib inline

## Input parameters

Select the date interval in which tiles will be processed:

In [8]:
default_from = datetime.date(year=2024, month=9, day=27)
default_to = datetime.date(year=2024, month=9, day=30)

from_date, to_date = notebook_utils.pick_date_range(default_from, default_to)

DatePicker(value=datetime.date(2024, 9, 27), description='From day', step=1)

DatePicker(value=datetime.date(2024, 9, 30), description='To day', step=1)

Select the tile's cloud percentage threshold:

In [9]:
percentage = notebook_utils.pick_percentage_slider()

IntSlider(value=15, continuous_update=False, description='Percentage of cloudiness')

## Find tiles

Select the area which delimites the tiles you want to process (left click to mark a point in the map, right click to erase current selection):

In [10]:
map_region = notebook_utils.MapRegion(center=(39.60595289727246, -122.82804126978336))

Map(center=[39.60595289727246, -122.82804126978336], controls=(ZoomControl(options=['position', 'zoom_in_text'…

In [11]:
coords = []
lats = []
lons = []
points = []

for value in map_region.get_region()[:-1]:
    coords.append(value)
    lats.append(value[1])
    lons.append(value[0])

start_date = from_date.value  # Start date to search images
end_date = to_date.value  # End date to search images
what = ['B04', 'B08']  # What we want to download
cc = percentage.value  # Minimum cloud cover on each image, 25 is 25%

for lon, lat in zip(lons, lats):
    points.append([lon, lat])
    print([lon, lat], start_date, end_date, what, cc)

In [12]:
import math

def distance(origin, destination):
    lat1, lon1 = origin
    lat2, lon2 = destination
    radius = 6371  # km

    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) * math.sin(dlat / 2) +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) *
         math.sin(dlon / 2) * math.sin(dlon / 2))
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    d = radius * c

    return d

In [13]:
i, p = 0, 0

while i != len(points):
    p = i + 1
    while p != len(points):
        dis = distance(points[i], points[p])
        divisions = int(dis / 100)
        # If the zones are separated by more than 100 km, generate intermediate zones
        if divisions > 0:
            toSum = [(points[i][0] - points[p][0]) / (divisions + 1) , (points[i][1] - points[p][1]) / (divisions + 1)]
            while divisions != 0:
                point = points[i][0] - (toSum[0] * divisions)
                # Not add duplicated lons/lats
                if point not in lons:
                    lons.append(points[i][0] - (toSum[0] * divisions))
                    lats.append(points[i][1] - (toSum[1] * divisions))
                divisions = divisions - 1
        p = p + 1 
    i = i + 1

In [14]:
start_date = from_date.value  # Start date to search images
end_date = to_date.value  # End date to search images
what = ['B04', 'B08']  # What we want to download
cc = 80  # Minimum cloud cover on each image, 25 is 25% (15 by default)

In [15]:
cc

80

In [16]:
# Demonstration: Californa tile coords
cali_coords = [
    [38.510161585585045, -122.99194335937501],
    [36.071996052851325, -121.25610351562501],
    [36.96374622851412, -121.46484375000001],
    [37.575739257598414, -121.55273437500001],
    [39.15202827678992, -122.62939453125001],
    [39.703620879017976, -123.12377929687501],
    [36.74397383313428, -119.94873046875001],
    [38.472809653752314, -121.60766601562501]
]

## Get Sentinel-2 packages

In [17]:
scenes_f1 = []
scenes_f2 = []

# To use the demonstration tile coords, coment this line to use teh coords obtained from the map before
coords = cali_coords

for latency, longitude in coords:
    try:
        # Get scenes from intital date
        f1 = s2froms3.get_scene_list(lon=longitude, lat=latency, start_date=start_date, end_date=start_date,
        what=what, cloud_cover_le=cc)

        # Get scenes from end date
        f2 = s2froms3.get_scene_list(lon=longitude, lat=latency, start_date=end_date, end_date=end_date,
        what=what, cloud_cover_le=cc)

        # Not add duplicated scenes
        if len(scenes_f1) == 0 or f1 not in scenes_f1:
            scenes_f1.append(f1)
            scenes_f2.append(f2)
    except Exception:
        pass

if len(scenes_f1) == 0:
    raise Exception('No data found')

In [18]:
scene = scenes_f1[-1][-1]
scene

'sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FH/2024/9/S2A_10SFH_20240927_0_L2A/B08.tif'

In [19]:
scene_band = rasterio.open('s3://'+scene)
windows = list(scene_band.block_windows())

In [20]:
tile_band_keys = [tup for tup in scenes_f1]
flat_paths = [path for sublist in tile_band_keys for path in sublist]

In [21]:
fexec = lithops.FunctionExecutor(
    runtime_memory=1024,
    log_level='INFO'
)
fexec.config['max_workers'] = 300

2025-06-19 21:36:17,847 [INFO] config.py:146 -- Lithops v3.6.1.dev0 - Python3.10
2025-06-19 21:36:20,171 [INFO] aws_s3.py:59 -- S3 client created - Region: us-east-1
2025-06-19 21:36:20,431 [INFO] aws_lambda.py:97 -- AWS Lambda client created - Region: us-east-1


In [22]:
flat_paths

['sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B04.tif',
 'sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B08.tif',
 'sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FF/2024/9/S2A_10SFF_20240927_0_L2A/B04.tif',
 'sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FF/2024/9/S2A_10SFF_20240927_0_L2A/B08.tif',
 'sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FF/2024/9/S2A_10SFF_20240927_1_L2A/B04.tif',
 'sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FF/2024/9/S2A_10SFF_20240927_1_L2A/B08.tif',
 'sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FG/2024/9/S2A_10SFG_20240927_0_L2A/B04.tif',
 'sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FG/2024/9/S2A_10SFG_20240927_0_L2A/B08.tif',
 'sentinel-cogs/sentinel-s2-l2a-cogs/11/S/KA/2024/9/S2A_11SKA_20240927_0_L2A/B04.tif',
 'sentinel-cogs/sentinel-s2-l2a-cogs/11/S/KA/2024/9/S2A_11SKA_20240927_0_L2A/B08.tif',
 'sentinel-cogs/sentinel-s2-l2a-cogs/11/S/KA/2024/9/S2A_11SKA_20240927_1_L2A/B04.tif',
 'sentinel-cogs/sentinel-s2-l2a-cogs/11/S/K

In [23]:
!pip install nbformat


In [24]:
%run /work/wrapper-function.ipynb


In [31]:
def get_tile_meta(input_data):
    import rasterio
    key = input_data["uri"]
    with rasterio.open('s3://' + key) as src:
        x1, y1 = src.profile['transform'] * (0, 0)
        x2, y2 = src.profile['transform'] * (src.profile['width'], src.profile['height'])
        return key, (x1, y1), (x2, y2)  


In [32]:
print("🔍 Estimating optimal memory per tile...")
estimates = []

for uri in flat_paths:
    est = estimate_chunk_size_local(uri)
    estimates.append(est)
    print(f"- {uri}")
    print(f"    → width: {est['width']}, height: {est['height']}, total_pixels: {est['total_pixels']}")
    print(f"    → best_chunk: {est['best_chunk']}, runtime_memory: {est['runtime_memory']} MB\n")


# Group by memory size
grouped_inputs = {}
for est in estimates:
    mem = est["runtime_memory"]
    grouped_inputs.setdefault(mem, []).append(est["uri"])

# Run get_tile_meta using grouped memory settings
tiles_meta = []
for mem, uris in grouped_inputs.items():
    print(f"🚀 Processing {len(uris)} tiles with {mem}MB memory...")
    fexec = lithops.FunctionExecutor(runtime_memory=mem)
    fexec.config["log_level"] = "INFO"
    fexec.config["keep_alive"] = True
    wrapped_inputs = [{"input_data": {"uri": uri}} for uri in uris]
    futures = fexec.map(get_tile_meta, wrapped_inputs)
    tiles_meta.extend(fexec.get_result(futures))

print("✅ Tile metadata extraction complete.")

🔍 Estimating optimal memory per tile...
- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B04.tif
    → width: 10980, height: 10980, total_pixels: 120560400
    → best_chunk: 6, runtime_memory: 512 MB

- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B08.tif
    → width: 10980, height: 10980, total_pixels: 120560400
    → best_chunk: 6, runtime_memory: 512 MB

- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FF/2024/9/S2A_10SFF_20240927_0_L2A/B04.tif
    → width: 10980, height: 10980, total_pixels: 120560400
    → best_chunk: 6, runtime_memory: 512 MB

- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FF/2024/9/S2A_10SFF_20240927_0_L2A/B08.tif
    → width: 10980, height: 10980, total_pixels: 120560400
    → best_chunk: 6, runtime_memory: 512 MB

- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FF/2024/9/S2A_10SFF_20240927_1_L2A/B04.tif
    → width: 10980, height: 10980, total_pixels: 120560400
    → best_chunk: 6, runtime_memory: 512 MB

- sentinel-cogs

2025-06-19 21:43:27,767 [INFO] config.py:146 -- Lithops v3.6.1.dev0 - Python3.10


- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FH/2024/9/S2A_10SFH_20240927_0_L2A/B08.tif
    → width: 10980, height: 10980, total_pixels: 120560400
    → best_chunk: 6, runtime_memory: 512 MB

🚀 Processing 14 tiles with 512MB memory...


2025-06-19 21:43:28,313 [INFO] aws_s3.py:59 -- S3 client created - Region: us-east-1
2025-06-19 21:43:28,593 [INFO] aws_lambda.py:97 -- AWS Lambda client created - Region: us-east-1
2025-06-19 21:43:28,822 [INFO] invokers.py:119 -- ExecutorID e9e20e-3 | JobID M000 - Selected Runtime: runtime-c2a280ba-8a05-4465-ac20-f8b77e0e377c:17a8db77-cf4e-4e60-a83d-b88c66e2d726-amd64 - 512MB
2025-06-19 21:43:28,875 [INFO] invokers.py:188 -- ExecutorID e9e20e-3 | JobID M000 - Starting function invocation: get_tile_meta() - Total: 14 activations
2025-06-19 21:43:29,166 [INFO] invokers.py:227 -- ExecutorID e9e20e-3 | JobID M000 - View execution logs at /tmp/lithops-root/logs/e9e20e-3-M000.log
2025-06-19 21:43:29,239 [INFO] executors.py:507 -- ExecutorID e9e20e-3 - Getting results from 14 function activations
2025-06-19 21:43:29,256 [INFO] wait.py:101 -- ExecutorID e9e20e-3 - Waiting for 14 function activations to complete


    0%|          | 0/14  

2025-06-19 21:43:34,115 [INFO] executors.py:631 -- ExecutorID e9e20e-3 - Cleaning temporary data


✅ Tile metadata extraction complete.


In [33]:
tiles_meta

[('sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B04.tif',
  (600000.0, 4000020.0),
  (709800.0, 3890220.0)),
 ('sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B08.tif',
  (600000.0, 4000020.0),
  (709800.0, 3890220.0)),
 ('sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FF/2024/9/S2A_10SFF_20240927_0_L2A/B04.tif',
  (600000.0, 4100040.0),
  (709800.0, 3990240.0)),
 ('sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FF/2024/9/S2A_10SFF_20240927_0_L2A/B08.tif',
  (600000.0, 4100040.0),
  (709800.0, 3990240.0)),
 ('sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FF/2024/9/S2A_10SFF_20240927_1_L2A/B04.tif',
  (600000.0, 4100040.0),
  (709800.0, 3990240.0)),
 ('sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FF/2024/9/S2A_10SFF_20240927_1_L2A/B08.tif',
  (600000.0, 4100040.0),
  (709800.0, 3990240.0)),
 ('sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FG/2024/9/S2A_10SFG_20240927_0_L2A/B04.tif',
  (600000.0, 4200000.0),
  (709800.0, 4090200.0)),
 ('sentinel-cogs/sentinel-s

In [37]:
regions = [(tile_id, bound1, bound2,
            int(tile_id.split('/')[7].split('_')[1][:2]),
            True) for tile_id, bound1, bound2 in tiles_meta]

# notebook_utils.MapRegion(regions=regions, center=(38.141080, -122.126583), zoom=6)

In [38]:
regions

[('sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B04.tif',
  (600000.0, 4000020.0),
  (709800.0, 3890220.0),
  10,
  True),
 ('sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B08.tif',
  (600000.0, 4000020.0),
  (709800.0, 3890220.0),
  10,
  True),
 ('sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FF/2024/9/S2A_10SFF_20240927_0_L2A/B04.tif',
  (600000.0, 4100040.0),
  (709800.0, 3990240.0),
  10,
  True),
 ('sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FF/2024/9/S2A_10SFF_20240927_0_L2A/B08.tif',
  (600000.0, 4100040.0),
  (709800.0, 3990240.0),
  10,
  True),
 ('sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FF/2024/9/S2A_10SFF_20240927_1_L2A/B04.tif',
  (600000.0, 4100040.0),
  (709800.0, 3990240.0),
  10,
  True),
 ('sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FF/2024/9/S2A_10SFF_20240927_1_L2A/B08.tif',
  (600000.0, 4100040.0),
  (709800.0, 3990240.0),
  10,
  True),
 ('sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FG/2024/9/S2A_10SFG_20240927_0_L2A/B04.

In [39]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import rasterio

def calculate_ndvi(scene, ij_window, storage):
    ij, window = ij_window
    band_4_s3_path = scene[0]  # Red band
    band_8_s3_path = scene[1]  # NIR band

    band_path_parts = band_4_s3_path.split('/')
    scene_id = band_path_parts[7]
    ndvi_filename = f'/tmp/{scene_id}_{ij}_NDVI.tif'
    ndvi_jpg_filename = f'/tmp/{scene_id}_{ij}_NDVI.jpg'

    # Open bands (Red and NIR)
    band4 = rasterio.open('s3://' + band_4_s3_path)
    band8 = rasterio.open('s3://' + band_8_s3_path)

    profile = band4.profile
    profile.update(dtype='float64', width=window.width, height=window.height)

    with rasterio.open(ndvi_filename, 'w', **profile) as dst:
        red = band4.read(1, window=window).astype('float64')
        nir = band8.read(1, window=window).astype('float64')
        with np.errstate(divide='ignore', invalid='ignore'):
            ndvi = np.where((nir + red) == 0, 0, (nir - red) / (nir + red)).astype('float64')
        ndvi_mean = np.mean(ndvi, axis=0)
        dst.write(ndvi, 1)

        # Set fixed scale for JPG visualization
        ndvi[0][0] = -1
        ndvi[0][1] = 1
        plt.imsave(ndvi_jpg_filename, ndvi, cmap="RdYlGn")

    with open(ndvi_jpg_filename, 'rb') as jpg_file:
        co_ndvi_jpg = storage.put_cloudobject(jpg_file.read(), key=ndvi_jpg_filename.replace('/tmp/', ''))

    return ndvi_filename, ndvi_mean, co_ndvi_jpg


def compute_ndvi_diff(old_scene, new_scene, ij_window, storage):
    ij, window = ij_window
    new_band_parts = new_scene[0].split('/')
    scene_id = new_band_parts[7]
    diff_jpg_filename = f'/tmp/{scene_id}_{ij}_NDVI_DIFF.jpg'
    result_key = old_scene[0].split('/')[7].rsplit('_', 3)[0]

    ndvi_file_old, ndvi_mean_old, co_ndvi_jpg_old = calculate_ndvi(old_scene, ij_window, storage)
    ndvi_file_new, ndvi_mean_new, co_ndvi_jpg_new = calculate_ndvi(new_scene, ij_window, storage)

    ndvi_old = rasterio.open(ndvi_file_old)
    ndvi_new = rasterio.open(ndvi_file_new)

    profile = ndvi_old.profile
    profile.update(dtype='float64', width=window.width, height=window.height)

    old_data = ndvi_old.read(1).astype('float64')
    new_data = ndvi_new.read(1).astype('float64')
    ndvi_diff = ((new_data - old_data) * (new_data + old_data)).astype('float64')

    # Set fixed scale for JPG visualization
    ndvi_diff[0][0] = -1
    ndvi_diff[0][1] = 1
    plt.imsave(diff_jpg_filename, ndvi_diff, cmap="RdYlGn")

    with open(diff_jpg_filename, 'rb') as diff_file:
        co_diff_jpg = storage.put_cloudobject(diff_file, key=diff_jpg_filename.replace('/tmp/', ''))

    return result_key, ij_window, co_ndvi_jpg_old, co_ndvi_jpg_new, co_diff_jpg


Using the selected parameters, get the identifiers of the selected tiles from Sentinel-2:

In [25]:
fexec = lithops.FunctionExecutor(
    runtime_memory=1024,
    log_level='INFO'
)
fexec.config['max_workers'] = 300


2025-06-19 07:13:21,729 [INFO] config.py:146 -- Lithops v3.6.1.dev0 - Python3.10


2025-06-19 07:13:22,161 [INFO] aws_s3.py:59 -- S3 client created - Region: us-east-1
2025-06-19 07:13:22,589 [INFO] aws_lambda.py:97 -- AWS Lambda client created - Region: us-east-1


In [40]:
iterdata = []

for scene_f1, scene_f2 in zip(scenes_f1, scenes_f2):
    # Only process if both lists have at least one element
    if scene_f1 and scene_f2:
        for window in windows:
            iterdata.append((scene_f1, scene_f2, window))

iterdata


[(['sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B04.tif',
   'sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B08.tif'],
  ['sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240930_0_L2A/B04.tif',
   'sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240930_0_L2A/B08.tif'],
  ((0, 0), Window(col_off=0, row_off=0, width=1024, height=1024))),
 (['sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B04.tif',
   'sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B08.tif'],
  ['sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240930_0_L2A/B04.tif',
   'sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240930_0_L2A/B08.tif'],
  ((0, 1), Window(col_off=1024, row_off=0, width=1024, height=1024))),
 (['sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B04.tif',
   'sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2

In [43]:
print("🔍 Estimating memory for NDVI diff tasks...")
ndvi_estimates = []
uri_cache = {}

for item in iterdata:
    uri = item[0][0]  # Use B04 from t1 for memory estimation

    if uri not in uri_cache:
        uri_cache[uri] = estimate_chunk_size_local(uri)

    est = uri_cache[uri]

    ndvi_estimates.append({
        "input_data": item,  # 3-tuple: (old_scene, new_scene, ij_window)
        "runtime_memory": est["runtime_memory"]
    })

    print(f"- {uri} → memory: {est['runtime_memory']} MB | best_chunk: {est['best_chunk']}")

# Group by memory
grouped_inputs = {}
for est in ndvi_estimates:
    mem = est["runtime_memory"]
    grouped_inputs.setdefault(mem, []).append(est["input_data"])

# Launch with Lithops
print("\n⚙️ Running compute_ndvi_diff with optimized memory grouping...")
results = []

for mem, inputs in grouped_inputs.items():
    print(f"🚀 Running {len(inputs)} tiles with {mem}MB...")
    fexec = lithops.FunctionExecutor(runtime_memory=mem)
    fexec.config["log_level"] = "INFO"
    fexec.config["keep_alive"] = True

    # ✅ Don’t wrap in input_data — match function signature
    fs = fexec.map(compute_ndvi_diff, inputs)
    results.extend(fexec.get_result(fs))

print("✅ NDVI diff computation complete.")


🔍 Estimating memory for NDVI diff tasks...
- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B04.tif → memory: 512 MB | best_chunk: 6
- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B04.tif → memory: 512 MB | best_chunk: 6
- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B04.tif → memory: 512 MB | best_chunk: 6
- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B04.tif → memory: 512 MB | best_chunk: 6
- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B04.tif → memory: 512 MB | best_chunk: 6
- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B04.tif → memory: 512 MB | best_chunk: 6
- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B04.tif → memory: 512 MB | best_chunk: 6
- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B04.tif → memory: 512 MB | best_chunk: 6
- sentinel-co

2025-06-19 21:56:52,115 [INFO] config.py:146 -- Lithops v3.6.1.dev0 - Python3.10


- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FH/2024/9/S2A_10SFH_20240927_0_L2A/B04.tif → memory: 512 MB | best_chunk: 6
- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FH/2024/9/S2A_10SFH_20240927_0_L2A/B04.tif → memory: 512 MB | best_chunk: 6
- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FH/2024/9/S2A_10SFH_20240927_0_L2A/B04.tif → memory: 512 MB | best_chunk: 6
- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FH/2024/9/S2A_10SFH_20240927_0_L2A/B04.tif → memory: 512 MB | best_chunk: 6
- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FH/2024/9/S2A_10SFH_20240927_0_L2A/B04.tif → memory: 512 MB | best_chunk: 6
- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FH/2024/9/S2A_10SFH_20240927_0_L2A/B04.tif → memory: 512 MB | best_chunk: 6
- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FH/2024/9/S2A_10SFH_20240927_0_L2A/B04.tif → memory: 512 MB | best_chunk: 6
- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FH/2024/9/S2A_10SFH_20240927_0_L2A/B04.tif → memory: 512 MB | best_chunk: 6
- sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FH/2024/9/S2A_

2025-06-19 21:56:52,701 [INFO] aws_s3.py:59 -- S3 client created - Region: us-east-1
2025-06-19 21:56:53,411 [INFO] aws_lambda.py:97 -- AWS Lambda client created - Region: us-east-1
2025-06-19 21:56:55,027 [INFO] invokers.py:119 -- ExecutorID e9e20e-5 | JobID M000 - Selected Runtime: runtime-c2a280ba-8a05-4465-ac20-f8b77e0e377c:17a8db77-cf4e-4e60-a83d-b88c66e2d726-amd64 - 512MB
2025-06-19 21:56:55,693 [INFO] invokers.py:188 -- ExecutorID e9e20e-5 | JobID M000 - Starting function invocation: compute_ndvi_diff() - Total: 605 activations
2025-06-19 21:56:59,236 [INFO] invokers.py:227 -- ExecutorID e9e20e-5 | JobID M000 - View execution logs at /tmp/lithops-root/logs/e9e20e-5-M000.log
2025-06-19 21:57:00,204 [INFO] executors.py:507 -- ExecutorID e9e20e-5 - Getting results from 605 function activations
2025-06-19 21:57:00,223 [INFO] wait.py:101 -- ExecutorID e9e20e-5 - Waiting for 605 function activations to complete


    0%|          | 0/605  

2025-06-19 21:57:38,109 [INFO] executors.py:631 -- ExecutorID e9e20e-5 - Cleaning temporary data


✅ NDVI diff computation complete.


In [44]:
grouped_results = collections.defaultdict(list)

for res in results:
    key, ij_window, co_jpg_f1, co_jpg_f2, co_jpg_diff = res
    grouped_results[key].append((ij_window, co_jpg_f1, co_jpg_f2, co_jpg_diff))

In [45]:
grouped_results.keys()

dict_keys(['S2A_10SFE', 'S2A_10SFF', 'S2A_10SFG', 'S2A_11SKA', 'S2A_10SFH'])

## Get and plot the computed jpg diff tile image

In [46]:
def get_jpg(data):
    if not data:
        print("Data is empty! Skipping this case...")
        return None  # Avoid proceeding if data is empty

    file_name = '_'.join(data[0][1].key.split('_')[:5])
    
    if 'DIFF' in data[0][1].key:
        output_file = f'AwsData/{file_name}_NDVI_DIFF.jpg'
    else:
        output_file = f'AwsData/{file_name}_NDVI.jpg'
        
    jpg_tiles = {}

    def load_tile(data):
        ij_window, co_jpg = data
        row = ij_window[0][0]
        col = ij_window[0][1]
        jpg_stream = fexec.storage.get_cloudobject(co_jpg, stream=True)

        if row not in jpg_tiles:
            jpg_tiles[row] = [None] * 11

        jpg_tiles[row][col] = Image.open(jpg_stream)

    with ThreadPoolExecutor(max_workers=16) as executor:
        futures = list(executor.map(load_tile, data))

    composite_image = Image.new('RGB', (scene_band.width, scene_band.height))

    x_offset = 0
    y_offset = 0

    for row in sorted(jpg_tiles.keys()):
        for img in jpg_tiles[row]:
            composite_image.paste(img, (x_offset, y_offset))
            x_offset += img.size[0]
        x_offset = 0
        y_offset += img.size[1]
        
    thumbnail_size = (640, 640)
    composite_image.thumbnail(thumbnail_size)

    images[output_file] = composite_image
    return output_file  # Return filename to confirm successful generation


In [47]:
grouped_results

defaultdict(list,
            {'S2A_10SFE': [(((0, 0),
                Window(col_off=0, row_off=0, width=1024, height=1024)),
               <lithops.storage.utils.CloudObject at 0x7d92ebfc7280>),
              (((0, 1),
                Window(col_off=1024, row_off=0, width=1024, height=1024)),
               <lithops.storage.utils.CloudObject at 0x7d92ebfc7430>),
              (((0, 2),
                Window(col_off=2048, row_off=0, width=1024, height=1024)),
               <lithops.storage.utils.CloudObject at 0x7d92ec5bc070>),
              (((0, 3),
                Window(col_off=3072, row_off=0, width=1024, height=1024)),
               <lithops.storage.utils.CloudObject at 0x7d92e96fa3b0>),
              (((0, 4),
                Window(col_off=4096, row_off=0, width=1024, height=1024)),
               <lithops.storage.utils.CloudObject at 0x7d92ec5da380>),
              (((0, 5),
                Window(col_off=5120, row_off=0, width=1024, height=1024)),
               <lithops

In [48]:
# Choose the correct key from the available ones
zone_key = 'S2A_10SFE'

if zone_key in grouped_results:
    group = grouped_results[zone_key]

    co_jpgs_f1 = [(name, f1) for name, f1, f2, diff in group]
    co_jpgs_f2 = [(name, f2) for name, f1, f2, diff in group]
    co_jpgs_diff = [(name, diff) for name, f1, f2, diff in group]

    # Display the lists if needed
    print("F1:", co_jpgs_f1)
    print("F2:", co_jpgs_f2)
    print("Diff:", co_jpgs_diff)
else:
    print(f"The key '{zone_key}' is not in grouped_results. Available keys: {list(grouped_results.keys())}")


F1: [(((0, 0), Window(col_off=0, row_off=0, width=1024, height=1024)), <lithops.storage.utils.CloudObject object at 0x7d92ec1c53f0>), (((0, 1), Window(col_off=1024, row_off=0, width=1024, height=1024)), <lithops.storage.utils.CloudObject object at 0x7d92ec1c6740>), (((0, 2), Window(col_off=2048, row_off=0, width=1024, height=1024)), <lithops.storage.utils.CloudObject object at 0x7d92ebfc6cb0>), (((0, 3), Window(col_off=3072, row_off=0, width=1024, height=1024)), <lithops.storage.utils.CloudObject object at 0x7d92e96f8df0>), (((0, 4), Window(col_off=4096, row_off=0, width=1024, height=1024)), <lithops.storage.utils.CloudObject object at 0x7d92eb4532e0>), (((0, 5), Window(col_off=5120, row_off=0, width=1024, height=1024)), <lithops.storage.utils.CloudObject object at 0x7d92ec2b9480>), (((0, 6), Window(col_off=6144, row_off=0, width=1024, height=1024)), <lithops.storage.utils.CloudObject object at 0x7d92e98b30a0>), (((0, 7), Window(col_off=7168, row_off=0, width=1024, height=1024)), <lith

In [ ]:
images = {}
with ThreadPoolExecutor(max_workers=3) as ex:
    fs = list(ex.map(get_jpg, [co_jpgs_f1, co_jpgs_f2, co_jpgs_diff]))

f, ax = plt.subplots(1,3, figsize=(18, 18))
i = 0
for j in sorted(images.keys()):
    ax[i].set_title(j)
    ax[i].imshow(images[j])
    i = i+1
plt.show()

## KPIs

In [34]:
import boto3

s3client = boto3.client('s3')
total_sz = 0

for scenes in [scenes_f1, scenes_f2]:
    for scene in scenes:
        # Ensure 'scene' is actually a list or tuple of paths
        if isinstance(scene, list) or isinstance(scene, tuple):
            for band_path in scene:
                if isinstance(band_path, str) and '/' in band_path:
                    bucket, key = band_path.split('/', 1)
                    meta = s3client.head_object(Bucket=bucket, Key=key)
                    total_sz += int(meta['ResponseMetadata']['HTTPHeaders']['content-length'])

# Assuming you already have the execution statistics
stats = [f.stats for f in fexec.futures if hasattr(f, 'stats') and 'worker_func_exec_time' in f.stats]
mean_exec_time = np.mean([stat['worker_func_exec_time'] for stat in stats])
throughput = (total_sz / 1_000_000_000) / mean_exec_time

print(f"Throughput: {throughput:.4f} GB/s")


Throughput: 0.9821 GB/s


In [35]:
print(f'Procesed {round(total_sz / 1_000_000_000, 2)} GB in {round(mean_exec_time, 2)} s => {round(throughput, 2)} GB/s')

Procesed 3.74 GB in 3.81 s => 0.98 GB/s


In [36]:
gbxms_price = 0.0000000167
sum_total_time = sum([stat['worker_exec_time'] for stat in stats]) * 1000
price = gbxms_price * sum_total_time * 1  # Price GB/ms * sum of times in ms * 1 GB

In [37]:
print(f'Experiment total price is {round(price, 3)} USD')

Experiment total price is 0.071 USD
